# 01B — Common canonical adapter

**Outcome:** convert any valid sector Pack into the same episode-aware
`SPEC-CORE`, with `SPLITS` and physically separate `SPEC-EVAL`.

The production path contains no ONT, splitter, well, valve or native metric
logic. Sector notebooks own native translation; this notebook only selects a
completed Pack and calls the common adapter.


## 1. Setup and sector switch

Run the relevant 01A notebook first. Change `SECTOR` to select its Pack.
`AS_OF_TS` is optional and limits only model-visible observations. The small
contract fixtures run in temporary directories and never scan the production
Pack again.


In [ ]:
import os
import resource
import sys
import tempfile
import time
import tracemalloc
from pathlib import Path

import pandas as pd
from IPython.display import display

try:
    import duckdb
except ImportError:
    !pip install -q duckdb
    import duckdb

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path(os.getenv(
    "ANOMALY_DRIVE_ROOT", "/content/drive/MyDrive/anomaly_detection"
))
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME", DRIVE_ROOT / "research" / "milestone1"
))
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from milestone1_core import (
    CORE_SCHEMAS,
    CORE_VERSION,
    EVAL_SCHEMAS,
    GAP_TOLERANCE_FACTOR,
    PACK_ENTITY_SCHEMA,
    PACK_EPISODE_SCHEMA,
    PACK_METRIC_SCHEMA,
    PACK_OBSERVATION_SCHEMA,
    build_canonical,
    check_core,
    core_fingerprint,
    read_json,
    save_pack,
)

SECTOR = os.getenv("ADAPTER_SECTOR", "telecom")  # "telecom" or "petrobras_3w"
BUILD_CANONICAL = os.getenv("BUILD_CANONICAL", "1") == "1"
RUN_CONTRACT_TESTS = os.getenv("RUN_CONTRACT_TESTS", "1") == "1"
AS_OF_TS = os.getenv("CANONICAL_AS_OF_TS") or None

PACK_RUN_IDS = {
    "telecom": "telecom_pack_v0_6_2",
    "petrobras_3w": "real_wells_expanded_v0_6_2",
}
CANONICAL_RUN_IDS = {
    "telecom": "telecom_core_v0_9_1_run1",
    "petrobras_3w": "petrobras_3w_core_v0_9_1_run1",
}
if SECTOR not in PACK_RUN_IDS:
    raise ValueError(f"Choose one of {list(PACK_RUN_IDS)}")

PACK_RUN_ID = os.getenv("ADAPTER_PACK_RUN_ID", PACK_RUN_IDS[SECTOR])
CANONICAL_RUN_ID = os.getenv("CANONICAL_RUN_ID", CANONICAL_RUN_IDS[SECTOR])
PACK_ROOT = DRIVE_ROOT / "outputs" / "packs" / SECTOR / PACK_RUN_ID
RUN_ROOT = (
    DRIVE_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}"
    / SECTOR / CANONICAL_RUN_ID
)

display(pd.Series({
    "sector": SECTOR,
    "pack_root": str(PACK_ROOT),
    "canonical_root": str(RUN_ROOT),
    "build": BUILD_CANONICAL,
    "as_of_ts": AS_OF_TS or "all observations",
    "contract_tests": RUN_CONTRACT_TESTS,
    "gap_tolerance_factor": GAP_TOLERANCE_FACTOR,
}, name="value").to_frame())


## 2. Materialise and validate once

`build_canonical()` validates the Pack and writes telemetry in bounded batches.
Entity bounds, episode bounds and collection gaps are derived without building
Python timestamp lists. `check_core()` then verifies schemas, row counts,
quality counts, the content fingerprint, global key uniqueness and references.


In [ ]:
display(pd.DataFrame([
    {"table": name, "columns": ", ".join(columns)}
    for name, columns in CORE_SCHEMAS.items()
]))

pack_manifest = read_json(PACK_ROOT / "pack_manifest.json")
runtime = {"elapsed_seconds": 0.0, "python_peak_mb": 0.0, "process_peak_mb": 0.0}

if BUILD_CANONICAL:
    tracemalloc.start()
    started = time.perf_counter()
    run_manifest = build_canonical(
        PACK_ROOT,
        RUN_ROOT,
        include_evaluation=True,
        as_of_ts=AS_OF_TS,
    )
    runtime["elapsed_seconds"] = time.perf_counter() - started
    _, python_peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    peak_rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    runtime["python_peak_mb"] = python_peak / (1024 ** 2)
    runtime["process_peak_mb"] = (
        peak_rss / (1024 ** 2) if sys.platform == "darwin" else peak_rss / 1024
    )
else:
    run_manifest = read_json(RUN_ROOT / "run_manifest.json")

core_audit = check_core(RUN_ROOT / "SPEC-CORE")
display(pd.Series(pack_manifest, name="value").to_frame())
display(pd.Series(runtime, name="value").to_frame())
display(pd.Series(core_audit, name="value").to_frame())


## 3. Small common-contract fixtures

These tests exercise the shared adapter without rebuilding the complete sector
Pack. One fixture proves all of the following:

- evaluation can be mounted or absent without changing `SPEC-CORE`;
- a deliberately leaky scorer fails when `SPEC-EVAL` is absent;
- one-second and five-second metrics keep independent timestamp grids;
- small timestamp jitter is not called a gap, while a missing scheduled
  observation is;
- observations after `as_of_ts` cannot change earlier canonical content;
- a duplicate key split across two Parquet parts is rejected.

The 01A original-versus-redacted test remains the primary translator-leakage
proof. This runtime test is deployment evidence.


In [ ]:
def write_contract_pack(destination, observations, *, evaluation=True, duplicate_part=False):
    destination = Path(destination)
    core = destination / "PACK-CORE"
    parts = core / "observations"
    parts.mkdir(parents=True)

    catalogue = pd.DataFrame([
        ("fast_signal", "test_asset", "gauge", "unit", "periodic", 1.0),
        ("slow_signal", "test_asset", "gauge", "unit", "periodic", 5.0),
    ], columns=PACK_METRIC_SCHEMA)
    observations[PACK_OBSERVATION_SCHEMA].to_parquet(
        parts / "part-00000.parquet", index=False
    )
    if duplicate_part:
        observations.iloc[[0]][PACK_OBSERVATION_SCHEMA].to_parquet(
            parts / "part-00001.parquet", index=False
        )
    catalogue.to_parquet(core / "metric_catalogue.parquet", index=False)
    pd.DataFrame([("asset-1", "test_asset")], columns=PACK_ENTITY_SCHEMA).to_parquet(
        core / "entity_registry.parquet", index=False
    )
    pd.DataFrame([
        ("asset-1::episode-1", "asset-1", "contract_fixture")
    ], columns=PACK_EPISODE_SCHEMA).to_parquet(
        core / "observation_episodes.parquet", index=False
    )

    evaluation_tables = []
    if evaluation:
        eval_root = destination / "PACK-EVAL"
        eval_root.mkdir()
        pd.DataFrame([{
            "entity_id": "asset-1",
            "start_ts": observations["event_ts"].min(),
            "end_ts": observations["event_ts"].max(),
            "condition_code": "fixture_condition",
            "label_source": "contract_fixture",
            "source_instance_id": "asset-1::episode-1",
        }])[EVAL_SCHEMAS["condition_states"]].to_parquet(
            eval_root / "condition_states.parquet", index=False
        )
        evaluation_tables = ["condition_states"]

    save_pack(
        destination,
        sector="contract_test",
        pack_version="common-adapter-v1",
        source_info={"source_id": "common-contract-fixture", "files": []},
        evaluation_tables=evaluation_tables,
    )


def deliberately_leaky_scorer(run_root):
    truth = Path(run_root) / "SPEC-EVAL"
    if not truth.is_dir():
        raise FileNotFoundError("SPEC-EVAL is not mounted")
    return sorted(path.name for path in truth.glob("*.parquet"))


def contract_observations():
    base = pd.Timestamp("2025-01-01 00:00:00", tz="UTC")
    fast = pd.DataFrame({
        "event_ts": [base + pd.Timedelta(seconds=value) for value in range(21)],
        "entity_id": "asset-1",
        "episode_id": "asset-1::episode-1",
        "metric_id": "fast_signal",
        "value": range(21),
        "quality_code": "measured",
    })
    slow = pd.DataFrame({
        "event_ts": [
            base,
            base + pd.Timedelta(seconds=5.2),
            base + pd.Timedelta(seconds=15),
            base + pd.Timedelta(seconds=20),
        ],
        "entity_id": "asset-1",
        "episode_id": "asset-1::episode-1",
        "metric_id": "slow_signal",
        "value": [20.0, 20.5, 21.0, None],
        "quality_code": ["measured", "measured", "measured", "invalid"],
    })
    return pd.concat([fast, slow], ignore_index=True)[PACK_OBSERVATION_SCHEMA]


In [ ]:
contract_status = "not_run"
if RUN_CONTRACT_TESTS:
    observations = contract_observations()
    cutoff = pd.Timestamp("2025-01-01 00:00:10", tz="UTC")
    assert observations["event_ts"].gt(cutoff).any()

    with tempfile.TemporaryDirectory() as temporary:
        temporary = Path(temporary)
        full_pack = temporary / "full_pack"
        truncated_pack = temporary / "truncated_pack"
        duplicate_pack = temporary / "duplicate_pack"
        write_contract_pack(full_pack, observations, evaluation=True)
        write_contract_pack(
            truncated_pack,
            observations.loc[observations["event_ts"].le(cutoff)].copy(),
            evaluation=False,
        )
        write_contract_pack(
            duplicate_pack, observations, evaluation=False, duplicate_part=True
        )

        mounted = temporary / "mounted"
        unmounted = temporary / "unmounted"
        build_canonical(full_pack, mounted, include_evaluation=True)
        build_canonical(full_pack, unmounted, include_evaluation=False)
        assert core_fingerprint(mounted / "SPEC-CORE") == core_fingerprint(
            unmounted / "SPEC-CORE"
        )
        deliberately_leaky_scorer(mounted)
        try:
            deliberately_leaky_scorer(unmounted)
        except FileNotFoundError:
            pass
        else:
            raise AssertionError("Negative control unexpectedly read unmounted truth")

        telemetry = pd.concat([
            pd.read_parquet(path)
            for path in sorted((mounted / "SPEC-CORE" / "telemetry").glob("part-*.parquet"))
        ], ignore_index=True)
        gaps = pd.read_parquet(mounted / "SPEC-CORE" / "collection_gaps.parquet")
        slow = telemetry.loc[telemetry["metric_id"].eq("slow_signal")]
        assert len(slow) == 4 and slow.iloc[-1]["quality_code"] == "invalid"
        assert len(gaps) == 1
        assert gaps.iloc[0]["gap_start"] == pd.Timestamp("2025-01-01 00:00:10.2", tz="UTC")
        assert gaps.iloc[0]["gap_end"] == pd.Timestamp("2025-01-01 00:00:15", tz="UTC")

        full_as_of = temporary / "full_as_of"
        truncated_as_of = temporary / "truncated_as_of"
        build_canonical(full_pack, full_as_of, include_evaluation=False, as_of_ts=cutoff)
        build_canonical(
            truncated_pack, truncated_as_of, include_evaluation=False, as_of_ts=cutoff
        )
        assert core_fingerprint(full_as_of / "SPEC-CORE") == core_fingerprint(
            truncated_as_of / "SPEC-CORE"
        )

        try:
            build_canonical(duplicate_pack, temporary / "duplicate_run", include_evaluation=False)
        except ValueError as error:
            assert "across Pack parts" in str(error)
        else:
            raise AssertionError("Cross-part duplicate key was accepted")

    contract_status = "pass"
    print("PASS — truth mounting cannot change SPEC-CORE")
    print("PASS — negative control fails without SPEC-EVAL")
    print("PASS — mixed cadence, jitter and missing observations are distinct")
    print("PASS — future observations cannot change an earlier as-of run")
    print("PASS — duplicate keys across Parquet parts are rejected")
else:
    print("NOT RUN — set RUN_CONTRACT_TESTS=1 to run the small common fixtures")


## 4. Acceptance and output inspection

The final section reports factual evidence. It does not use a hard-coded
“sector-neutral” Boolean: sector neutrality is demonstrated by the same
`build_canonical()` call accepting either valid Pack interface.


In [ ]:
acceptance = {
    "contract_version": CORE_VERSION,
    "sector": SECTOR,
    "pack_interface_version": pack_manifest["pack_interface_version"],
    "common_adapter_function": "build_canonical",
    "global_key_audit": "pass",
    "fingerprint_verified": core_audit["fingerprint_verified"],
    "contract_fixtures": contract_status,
    "evaluation_mounted": run_manifest["evaluation_mounted"],
    "tier0_tables": list(CORE_SCHEMAS),
}
display(pd.Series(acceptance, name="result").to_frame())

for name in (
    "metric_catalogue", "entity_registry",
    "observation_episodes", "collection_gaps",
):
    frame = pd.read_parquet(RUN_ROOT / "SPEC-CORE" / f"{name}.parquet")
    print(f"\n{name}: {len(frame):,} rows")
    display(frame.head(10))

telemetry_parts = sorted((RUN_ROOT / "SPEC-CORE" / "telemetry").glob("part-*.parquet"))
print(f"\ntelemetry: {len(telemetry_parts):,} parts")
display(pd.read_parquet(telemetry_parts[0]).head(10))

for name in pack_manifest.get("split_tables", []):
    frame = pd.read_parquet(RUN_ROOT / "SPLITS" / f"{name}.parquet")
    print(f"\nSPLITS/{name}: {len(frame):,} rows (not model input)")
    display(frame.head(10))

print("SPEC-EVAL tables:", pack_manifest.get("evaluation_tables", []))
display(pd.Series(run_manifest, name="value").to_frame())
print("Canonical run:", RUN_ROOT)
print("Next: 02_CANONICAL_EDA.ipynb")
